# 🚀 J.A.R.V.I.S. Autonomous Brain Fine-Tuning Studio
Train your customized J.A.R.V.I.S. model on Google Colab (Free T4 GPU) using Unsloth & QLoRA.
Fast, 2x memory efficient, and exports directly to GGUF for Ollama!

In [ ]:
# 1. Install Unsloth & SFT requirements
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model (Qwen2.5-Coder-3B or Llama-3.2-3B in 4-bit)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA parameters initialized!")

In [ ]:
# 4. Upload your dataset or paste SFT records
# Upload 'sft_alpaca.jsonl' from your Jarvis data/training/ folder
from google.colab import files
import json
from datasets import Dataset

uploaded = files.upload()
filename = list(uploaded.keys())[0]

data = []
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} training samples!")

In [ ]:
# 5. Execute Supervised Fine-Tuning (SFT)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="output",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir="jarvis_lora",
    ),
)
trainer.train()
print("Training Complete!")

In [ ]:
# 6. Export to 16-bit GGUF or Ollama Modelfile
model.save_pretrained_gguf("jarvis_gguf", tokenizer, quantization_method="q4_k_m")
# Download the GGUF model and drop it into Ollama:
# ollama create jarvis-custom -f Modelfile
print("Export complete!")